# 🛡️ Data Reliability System
**Detects problems in your data, explains why they happened, and suggests fixes.**

Run each cell from top to bottom by clicking the ▶ Play button on the left of each cell.

---

## Cell 1 — Install Required Packages
Run this first. It installs everything the tool needs.

In [ ]:
!pip install pandas pyyaml requests sqlalchemy apscheduler groq -q
print('✅ All packages installed')

## Cell 2 — Create Sample Data Files
This creates two example CSV files so you can test the tool right away.
Later you can upload your own files and point the tool at them.

In [ ]:
import os
import pandas as pd

os.makedirs('sample_data', exist_ok=True)

sales = pd.DataFrame({
    'date': ['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05',
             '2024-01-06','2024-01-07','2024-01-08','2024-01-09','2024-01-10'],
    'revenue': [1500.00, 2300.50, 800.00, 3200.75, 1100.00,
                950.00, 2750.00, 1800.25, 4200.00, 600.50],
    'customer_id': ['C001','C002','C003','C004','C005','C006','C007','C008','C009','C010'],
    'region': ['North','South','East','West','North','South','East','West','North','South']
})
sales.to_csv('sample_data/sales.csv', index=False)

customers = pd.DataFrame({
    'customer_id': ['C001','C002','C003','C004','C005'],
    'name': ['Alice Johnson','Bob Smith','Carol White','David Brown','Eva Green'],
    'email': ['alice@example.com','bob@example.com','carol@example.com', None, 'eva@example.com'],
    'signup_date': ['2023-06-01','2023-07-15','2023-08-20','2023-09-05','2023-10-12']
})
customers.to_csv('sample_data/customers.csv', index=False)

print('✅ Sample files created: sample_data/sales.csv and sample_data/customers.csv')
print()
print('Sales Data Preview:')
display(sales)
print('\nCustomer Data Preview:')
display(customers)

## Cell 3 — Build the Detection Engine
This is the "Doctor" — it reads your data and finds what's wrong.

In [ ]:
import json, sqlite3
from datetime import datetime

# ---------- HISTORY STORAGE ----------
DB_PATH = 'reliability_history.db'

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS snapshots (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_name TEXT, captured_at TEXT, row_count INTEGER,
            column_names TEXT, null_rates TEXT, numeric_stats TEXT, file_modified_at REAL
        )''')
    conn.commit()
    return conn

def save_snapshot(name, df, file_mod=0.0):
    col_names = list(df.columns)
    null_rates = {c: round(df[c].isna().mean()*100,2) for c in df.columns}
    numeric_stats = {}
    for c in df.select_dtypes(include='number').columns:
        numeric_stats[c] = {'min': float(df[c].min()) if not df[c].isna().all() else None,
                            'max': float(df[c].max()) if not df[c].isna().all() else None,
                            'mean': round(float(df[c].mean()),4) if not df[c].isna().all() else None}
    conn = get_conn()
    conn.execute('INSERT INTO snapshots (source_name,captured_at,row_count,column_names,null_rates,numeric_stats,file_modified_at) VALUES (?,?,?,?,?,?,?)',
        (name, datetime.utcnow().isoformat(), len(df), json.dumps(col_names), json.dumps(null_rates), json.dumps(numeric_stats), file_mod))
    conn.commit(); conn.close()

def get_last_snapshot(name):
    conn = get_conn()
    row = conn.execute('SELECT * FROM snapshots WHERE source_name=? ORDER BY id DESC LIMIT 1',(name,)).fetchone()
    conn.close()
    if not row: return None
    return {'row_count':row[3],'column_names':json.loads(row[4]),'null_rates':json.loads(row[5]),'numeric_stats':json.loads(row[6]),'file_modified_at':row[7]}

# ---------- DETECTOR ----------
def run_checks(df, last_snapshot, rules):
    issues = []
    current_cols = set(df.columns)
    required = set(rules.get('required_columns', []))
    missing_required = required - current_cols
    if missing_required:
        issues.append({'type':'missing_required_column','severity':'critical',
            'detail':f"Required column(s) not found: {', '.join(sorted(missing_required))}",
            'columns':list(missing_required)})
    min_rows = rules.get('min_row_count', 1)
    if len(df) < min_rows:
        issues.append({'type':'low_row_count','severity':'critical',
            'detail':f'Only {len(df)} rows found (minimum expected: {min_rows})',
            'current':len(df),'threshold':min_rows})
    max_null = rules.get('max_null_percent', 10)
    for col in df.columns:
        pct = round(df[col].isna().mean()*100, 2)
        if pct > max_null:
            issues.append({'type':'high_null_rate','severity':'warning',
                'detail':f"Column '{col}' has {pct}% missing values (limit: {max_null}%)",
                'column':col,'null_percent':pct})
    if last_snapshot:
        prev_cols = set(last_snapshot['column_names'])
        added = current_cols - prev_cols
        removed = prev_cols - current_cols
        if added:
            issues.append({'type':'schema_drift_added','severity':'warning',
                'detail':f"New column(s) appeared: {', '.join(sorted(added))}"})
        if removed:
            issues.append({'type':'schema_drift_removed','severity':'critical',
                'detail':f"Column(s) disappeared: {', '.join(sorted(removed))}"})
        prev_rows = last_snapshot['row_count']
        if prev_rows > 0:
            drop = (prev_rows - len(df)) / prev_rows * 100
            if drop > 20:
                issues.append({'type':'row_count_drop','severity':'critical',
                    'detail':f'Row count dropped {round(drop)}% (was {prev_rows}, now {len(df)})',
                    'drop_percent':round(drop,1)})
        for col in df.select_dtypes(include='number').columns:
            hist = last_snapshot['numeric_stats'].get(col)
            if not hist or hist['min'] is None: continue
            buf = max((hist['max']-hist['min'])*0.5, 1)
            cmin, cmax = float(df[col].min()), float(df[col].max())
            if cmin < hist['min']-buf or cmax > hist['max']+buf:
                issues.append({'type':'numeric_anomaly','severity':'warning',
                    'detail':f"Column '{col}': history [{hist['min']},{hist['max']}] vs now [{cmin},{cmax}]",
                    'column':col})
    return issues

# ---------- DIAGNOSER ----------
def diagnose(issues, last_snapshot, file_mod):
    file_changed = last_snapshot and file_mod > last_snapshot.get('file_modified_at', 0)
    for issue in issues:
        t = issue['type']
        if t in ('schema_drift_removed','missing_required_column'):
            issue['likely_cause'] = 'The source file was recently modified or re-exported' if file_changed else 'A column was accidentally deleted or renamed'
            issue['suggested_action'] = 'Re-open the source and make sure all required columns are present'
        elif t == 'schema_drift_added':
            issue['likely_cause'] = 'A new field was added to the source'
            issue['suggested_action'] = 'Verify the new column is intentional'
        elif t in ('low_row_count','row_count_drop'):
            issue['likely_cause'] = 'File may have been overwritten with partial data' if file_changed else 'Rows may have been accidentally deleted'
            issue['suggested_action'] = 'Check if the file export/import job completed successfully'
        elif t == 'high_null_rate':
            col = issue.get('column','?')
            issue['likely_cause'] = f"Some records are missing values for '{col}'"
            issue['suggested_action'] = f"Review the source data for '{col}'. Auto-fix will fill these if enabled."
        elif t == 'numeric_anomaly':
            col = issue.get('column','?')
            issue['likely_cause'] = f"Values in '{col}' are outside the range seen in previous scans"
            issue['suggested_action'] = 'Verify if this change is expected (e.g. a legitimate spike or data entry error)'
        else:
            issue['likely_cause'] = 'Unknown — requires manual investigation'
            issue['suggested_action'] = 'Review the data file and compare with the previous version'
    return issues

# ---------- REMEDIATOR ----------
def auto_fix(df, file_path):
    fix_log = []
    dupes = df.duplicated().sum()
    if dupes > 0:
        df = df.drop_duplicates()
        fix_log.append(f'Removed {dupes} duplicate row(s)')
    for col in df.select_dtypes(include='number').columns:
        n = df[col].isna().sum()
        if n > 0:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            fix_log.append(f"Filled {n} missing number(s) in '{col}' with median ({med})")
    for col in df.select_dtypes(include='object').columns:
        n = df[col].isna().sum()
        if n > 0:
            df[col] = df[col].fillna('UNKNOWN')
            fix_log.append(f"Filled {n} missing text value(s) in '{col}' with 'UNKNOWN'")
    if fix_log:
        backup = file_path.replace('.csv', f'_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')
        pd.read_csv(file_path)  # read original
        df.to_csv(file_path, index=False)
        fix_log.append(f'Fixed file saved')
    return df, fix_log

print('✅ Detection engine loaded')

## Cell 3b — 🤖 AI Diagnosis Engine (Optional)
Paste your Anthropic API key in the next cell to enable Claude-powered root cause analysis.
If you leave it blank, the tool uses built-in rule-based diagnosis instead — everything still works.

In [ ]:
from groq import Groq as _Groq
import requests as _requests

# ─── PASTE YOUR GROQ API KEY HERE ───────────────────────────────────
GROQ_API_KEY = ""        # Get free key at console.groq.com
# ─── PASTE YOUR SLACK WEBHOOK URL HERE ──────────────────────────────
SLACK_WEBHOOK_URL = ""   # e.g. "https://hooks.slack.com/services/..."
# ────────────────────────────────────────────────────────────────────

# ---------- SLACK ALERTS ----------
def send_slack(issues, fix_log, name, row_count):
    if not SLACK_WEBHOOK_URL:
        return
    if not issues:
        text = f"✅ *{name}* — All checks passed ({row_count} rows)"
    else:
        lines = [f"⚠️ *{name}* — {len(issues)} issue(s) found ({row_count} rows)\n"]
        for issue in issues:
            emoji = {"critical": "🔴", "warning": "🟡"}.get(issue.get("severity", "warning"), "🟡")
            lines.append(f"{emoji} {issue['detail']}")
            if "likely_cause" in issue:
                lines.append(f"   📌 *Likely cause:* {issue['likely_cause']}")
            if "suggested_action" in issue:
                lines.append(f"   💡 *What to do:* {issue['suggested_action']}")
            lines.append("")
        if fix_log:
            lines.append("🔧 *Auto-fixes applied:*")
            for f in fix_log:
                lines.append(f"   • {f}")
        text = "\n".join(lines)
    _requests.post(SLACK_WEBHOOK_URL, json={"text": text}, timeout=5)

# ---------- AI DIAGNOSIS ----------
def ai_diagnose(issues, last_snapshot, df, source_name):
    if not GROQ_API_KEY:
        return None
    client = _Groq(api_key=GROQ_API_KEY)

    context_parts = [f"Data source: {source_name}"]
    if last_snapshot:
        context_parts.append(f"Previous row count: {last_snapshot['row_count']}")
        context_parts.append(f"Previous columns: {', '.join(last_snapshot['column_names'])}")
        context_parts.append(f"Previous null rates: {json.dumps(last_snapshot.get('null_rates', {}))}")
        context_parts.append(f"Previous numeric ranges: {json.dumps(last_snapshot.get('numeric_stats', {}))}")
    else:
        context_parts.append("No previous snapshot — this is the first scan.")
    context_parts.append(f"Current data sample (first 5 rows): {json.dumps(df.head(5).to_dict(orient='records'))}")
    context = "\n".join(context_parts)

    prompt = f"""You are a data reliability expert. A monitoring system has detected problems in a data file.

Analyze the context below and for each issue provide:
1. A short, plain-English explanation of the most likely root cause (1-2 sentences)
2. A specific, actionable remediation step (1-2 sentences)

Be concrete — mention column names, row counts, and percentages where relevant.
Write as if explaining to someone who does not code.

=== DATA CONTEXT ===
{context}

=== DETECTED ISSUES ===
{json.dumps(issues, indent=2)}

Respond with a JSON array. Each element must match one issue (same order) and contain:
- "likely_cause": string
- "suggested_action": string

Return ONLY valid JSON — no markdown fences, no text outside the array."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    raw = response.choices[0].message.content.strip()

    if "```" in raw:
        parts = raw.split("```")
        raw = parts[1] if len(parts) >= 2 else raw
        if raw.startswith("json"):
            raw = raw[4:]

    start, end = raw.find("["), raw.rfind("]") + 1
    try:
        diagnoses = json.loads(raw[start:end]) if start != -1 else []
    except json.JSONDecodeError:
        diagnoses = []

    for i, issue in enumerate(issues):
        if i < len(diagnoses):
            issue["likely_cause"]     = diagnoses[i].get("likely_cause", "See data context")
            issue["suggested_action"] = diagnoses[i].get("suggested_action", "Review file manually")

    return issues

if GROQ_API_KEY:
    print("✅ AI diagnosis enabled — Llama 3 (Groq) will analyze issues")
else:
    print("ℹ️  No API key — rule-based diagnosis will be used (still works fine)")
if SLACK_WEBHOOK_URL:
    print("✅ Slack alerts enabled")
else:
    print("ℹ️  No Slack URL — alerts will print to terminal only")

## Cell 4 — Define Your Data Sources
Edit the list below to point at your own files. Add as many as you like.

In [ ]:
DATA_SOURCES = [
    {
        'name': 'Sales Data',
        'path': 'sample_data/sales.csv',
        'auto_fix': True,
        'rules': {
            'max_null_percent': 5,
            'min_row_count': 5,
            'required_columns': ['date', 'revenue', 'customer_id']
        }
    },
    {
        'name': 'Customer List',
        'path': 'sample_data/customers.csv',
        'auto_fix': False,
        'rules': {
            'max_null_percent': 10,
            'min_row_count': 3,
            'required_columns': ['customer_id', 'name', 'email']
        }
    }
]

print('✅ Data sources configured:', [s['name'] for s in DATA_SOURCES])

## Cell 5 — ▶ RUN THE CHECKS
This is the main cell. Run it anytime to check all your data sources.

In [ ]:
SEVERITY_EMOJI = {'critical': '🔴', 'warning': '🟡', 'info': '🔵'}

def print_report(name, issues, fix_log, row_count, used_ai=False):
    print(f'\n{"="*55}')
    print(f'  📋 {name}{"  🤖 AI-diagnosed" if used_ai and issues else ""}')
    print(f'{"="*55}')
    if not issues:
        print(f'  ✅ All checks passed! ({row_count} rows)')
    else:
        print(f'  ⚠️  {len(issues)} issue(s) found ({row_count} rows)\n')
        for issue in issues:
            emoji = SEVERITY_EMOJI.get(issue.get('severity','warning'), '🟡')
            print(f'  {emoji} {issue["detail"]}')
            if 'likely_cause' in issue:
                print(f'     📌 Likely cause:  {issue["likely_cause"]}')
            if 'suggested_action' in issue:
                print(f'     💡 What to do:    {issue["suggested_action"]}')
            print()
    if fix_log:
        print('  🔧 Auto-fixes applied:')
        for f in fix_log:
            print(f'     • {f}')

print(f'🚀 Running checks at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

for source in DATA_SOURCES:
    name = source['name']
    path = source['path']
    try:
        df = pd.read_csv(path)
        file_mod = os.path.getmtime(path)
        last = get_last_snapshot(name)
        issues = run_checks(df, last, source['rules'])
        used_ai = False
        if issues:
            ai_result = ai_diagnose(issues, last, df, name)
            if ai_result is not None:
                issues = ai_result
                used_ai = True
            else:
                issues = diagnose(issues, last, file_mod)
        fix_log = []
        if issues and source.get('auto_fix'):
            df, fix_log = auto_fix(df, path)
        print_report(name, issues, fix_log, len(df), used_ai)
        send_slack(issues, fix_log, name, len(df))
        save_snapshot(name, df, file_mod)
    except Exception as e:
        print(f'\n🔴 ERROR checking "{name}": {e}')

print(f'\n{"="*55}')
print('Done!')


## Cell 6 — 🧪 Test: Simulate a Problem (Optional)
Run this cell to intentionally break the sales data, then re-run Cell 5 to see the alerts fire.

In [ ]:
# This removes the 'revenue' column and reduces rows — simulating a bad export
df_broken = pd.read_csv('sample_data/sales.csv')
df_broken = df_broken.drop(columns=['revenue']).head(3)
df_broken.to_csv('sample_data/sales.csv', index=False)
print('⚠️ sales.csv has been deliberately broken.')
print('Now run Cell 5 again to see the alerts!')
display(df_broken)

## Cell 7 — 🔄 Restore Sample Data
Run this to undo the damage from Cell 6.

In [ ]:
sales = pd.DataFrame({
    'date': ['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05',
             '2024-01-06','2024-01-07','2024-01-08','2024-01-09','2024-01-10'],
    'revenue': [1500.00,2300.50,800.00,3200.75,1100.00,950.00,2750.00,1800.25,4200.00,600.50],
    'customer_id': ['C001','C002','C003','C004','C005','C006','C007','C008','C009','C010'],
    'region': ['North','South','East','West','North','South','East','West','North','South']
})
sales.to_csv('sample_data/sales.csv', index=False)
print('✅ sales.csv restored to original. Run Cell 5 again to confirm all-clear.')
display(sales)

---
## 📂 How to Use Your Own Data

1. Click **'Add data'** (top right in Kaggle) → Upload your CSV file
2. In **Cell 4**, change the `path` to your file path (Kaggle puts uploaded files in `/kaggle/input/your-dataset-name/yourfile.csv`)
3. Update `required_columns` to match your column names
4. Run Cell 5

That's it!